# 01 - Getting Started: Multi-Energy System Co-Simulation

Welcome to **`energysim`**! This tutorial introduces the fundamental concepts of multi-energy system (MES) co-simulation:

1. Initializing an `energysim.world` co-simulation canvas.
2. Macro time-steps ($t_{\text{macro}}$) vs. Micro time-steps ($t_{\text{micro}}$).
3. Adding physical component models (`BatteryBESS`, `PhysicsPV`).
4. Injecting dynamic boundary conditions (signals and profiles).
5. Establishing directional variable connections and automatic unit scaling.
6. Running the simulation and inspecting results in pandas DataFrames.


In [ ]:
from IPython.display import display
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from energysim import world
from energysim.models.renewables import PhysicsPV
from energysim.models.storage import BatteryBESS

print("energysim imported successfully!")


## Step 1: Initialize the Co-Simulation World

The `world` object orchestrates time-synchronization, variable routing, and coupling algorithms.
Here we configure a 24-hour simulation ($86,400\text{ s}$) with a 15-minute macro exchange step ($900\text{ s}$).


In [ ]:
sim = world(
    start_time=0,
    stop_time=86400,
    t_macro=900,
    power_unit="kW",  # Global default unit
    logging=False
)
print("Canvas initialized:", sim)


## Step 2: Instantiate Physical Components

We will add a **5 MW Solar PV Plant** and a **10 MWh / 2.5 MW Battery Energy Storage System (BESS)**.


In [ ]:
# 1. 5 MWp Solar PV array
pv = PhysicsPV(name="solar_pv", p_dc_stc_mw=5.0, inverter_p_ac_max_mw=4.5)

# 2. 10 MWh / 2.5 MW Battery
bess = BatteryBESS(name="battery", capacity_mwh=10.0, p_max_mw=2.5, soc_init=0.4)

# 3. Add to world canvas
sim.add_model("solar_pv", model=pv, outputs=["P_ac", "T_cell"])
sim.add_model("battery", model=bess, outputs=["P_actual", "SoC", "T_batt"])


## Step 3: Add Time-Varying Signals

We create synthetic diurnal profiles for solar irradiance ($W/m^2$) and battery charging dispatch commands.


In [ ]:
# Diurnal solar irradiance curve peaking at 900 W/m2 at noon
def solar_irradiance(t):
    hr = (t / 3600.0) % 24.0
    if 6.0 <= hr <= 18.0:
        return 900.0 * np.sin(np.pi * (hr - 6.0) / 12.0)
    return 0.0

# Charge battery on midday solar peak, discharge into evening load
def battery_dispatch(t):
    hr = (t / 3600.0) % 24.0
    if 10.0 <= hr <= 14.0:
        return 1.8   # Charge at 1.8 MW
    elif 18.0 <= hr <= 22.0:
        return -1.5  # Discharge at 1.5 MW
    return 0.0

sim.add_signal("irr_sig", signal=solar_irradiance)
sim.add_signal("bess_sig", signal=battery_dispatch, units={"y": "MW"})


## Step 4: Connect Variables

Route the solar irradiance into the PV model, and the dispatch command into the battery.


In [ ]:
sim.add_connections({
    "irr_sig.y": "solar_pv.G_poa",
    "bess_sig.y": "battery.P_cmd",
})
print("Connections established!")


## Step 5: Execute Simulation and Plot Results


In [ ]:
sim.simulate(pbar=True)
results = sim.last_results

df_pv = results["solar_pv"]
df_bess = results["battery"]

print("Simulation finished! Extracted DataFrames:")
display(df_pv.head())
display(df_bess.head())


In [ ]:
# Plotting Results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

hours = df_pv.index / 3600.0   # Results tables are indexed by time in seconds

ax1.plot(hours, df_pv["P_ac"], color="orange", lw=2, label="Solar PV (MW)")
ax1.plot(hours, df_bess["P_actual"], color="blue", lw=2, label="BESS Power (MW)")
ax1.set_ylabel("Power (MW)")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(hours, df_bess["SoC"] * 100, color="green", lw=2, label="Battery SoC (%)")
ax2.set_xlabel("Time of Day (Hours)")
ax2.set_ylabel("SoC (%)")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()
